In [ ]:
from pathlib import Path
import sys
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
 )

sys.path.insert(0, str(Path("..").resolve()))
from backend.registry import register_dataset, register_model

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(Path("..").resolve()))

In [ ]:
df = pd.read_csv('brfss_2024.zip', compression='zip')
pd.set_option('display.max_columns', None)

target_col = 'CVDSTRK3'

# Keep only rows where we have a clear stroke status
df = df[(df[target_col] == 1) | (df[target_col] == 2)]

# Convert the target to a binary outcome for model training: 1 = Stroke, 0 = No Stroke
df[target_col] = (df[target_col] == 1).astype(int)

print("Amount of rows and features:", df.shape)
df.head(1)

In [ ]:
# Manual feature selection - the goal is to drop features such as administrative and survey design variables.
# These variables tell how the data was collected, not what respondents reported about their health or behaviour.

df.drop(columns=[
    # When the survey was conducted and general identifiers whether is eligible to be included in the survey
    "FMONTH", "IDATE", "IMONTH", "IDAY", "IYEAR", "DISPCODE", "SEQNO", "_PSU", "SAFETIME",
    "LADULT1", "CADULT1", "RESPSLC1", "CTELENM1", "CTELNUM1","CELPHON1", "CELLFON5", "LANDLINE",
    "LANDSEX3", "CELLSEX3",
    
    # Residency and household variables
    "PVTRESD1", "PVTRESD3", "COLGHOUS", "STATERE1", "CSTATE1", "NUMADULT", 
    "HHADULT", "NUMHHOL4", "NUMPHON4", "CPDEMO1C", "CCLGHOUS", "GUNLOAD", 
    "LOADULK2", "FIREARM5", "_CHLDCNT", "CHILDREN", "CAGEG", "RENTHOM1",

    # Survey language and version variables
    "QSTVER", "QSTLANG", "ICFQSTVR",

    # Statistical sampling weights & design variables
    "_STSTR", "_STRWT", "_RAWRAKE", "_WT2RAKE", "_CLLCPWT", 
    "_DUALUSE", "_DUALCOR", "_LLCPWT2", "_LLCPWT",

    # Post Course of Treatment questions
    "CSRVINST", "CSRVINSR", "CSRVDEIN", "PCSTALK2", "CSRVSUM",

    # Caregiver - about the respondent's relationship to their caregiver, 
    # not about the respondent's health or behaviour.
    "CAREGIV1", "CRGVREL5", "CRGVPRB4", "CRGVALZD", 
    "CRGVNURS", "CRGVPER2", "CRGVHOU2", "CRGVHRS2", "CRGVLNG2", 

    # Family Planning
    "RCSRLTN2", "HADSEX", "PFPPRVN4", "TYPCNTR9", "NOBCUSE8",

    "RCSXBRTH", "RCSGEND1", "HPVDSHT", "CSRVCTL2",
], inplace=True)

print("Dataset after dropping columns that are unrelated to health outcomes by design:", df.shape)
df.head(1)

In [ ]:
# Find highly correlated features using Cramér's V for categorical variables.
# from feature_analysis import find_high_correlation_pairs

# pairs_df = find_high_correlation_pairs(df, threshold=0.85, sample_n = 50_000, random_state = 42)
# print(pairs_df.to_string(index=False))

In [ ]:
# Remove redundant features

df.drop(columns=[
    # General health / days unhealthy
    # Keep: GENHLTH, PHYSHLTH, MENTHLTH, _TOTINDA 
    # Precise amount of days unhealthy in the past 30 days, while the alternatives are coarser categories or binary variables
    "_RFHLTH", "_PHYS14D", "_MENT14D", "POORHLTH", "EXERANY2",

    # BMI / anthropometrics
    # Keep: _BMI5 (continuous computed BMI)
    # Raw height and weight in all unit encodings are redundant as we have _BMI5. 
    # _BMI5CAT and _RFBMI5 are redundant to _BMI5.
    "WEIGHT2", "HEIGHT3", "HTIN4", "HTM4", 
    "WTKG3", "_BMI5CAT", "_RFBMI5",

    # Race / ethnicity
    # Keep: _RACE (8-category combined variable covering all race variants)
    "_RACEG21", "_RACEGR3", "_RACEPRV", "_MRACE1", 
    "_IMPRACE", "_CHISPNC", "_CRACE1", "_HISPANC",

    # Age
    # Keep: _AGE80 (Alternatives are encoded in age groups, but _AGE80 is a continuous number 
    # that gives us more precise information about the respondent's age)
    "_AGE65YR", "_AGE_G", "_AGEG5YR", "_LCSAGE",

    # Smoking
    # Keep: _SMOKER3 (Derived variable that combines current / former / never / every day smoking status)
    # Keep: LCSNUMC_ (preciese number of cigarettes smoked per day)
    # Raw inputs SMOKE100 and SMOKDAY2 are fully absorbed by _SMOKER3.
    # _RFSMOK3 is a binary collapse of _SMOKER3. USENOW3, LASTSMK2,
    "_RFSMOK3", "USENOW3", "LASTSMK2", "STOPSMK2", "LCSNUMCG",
    "HEATTBCO", "SMOKE100", "SMOKDAY2", "_PACKDAY", "_LCSSMKG",
    "_PACKYRS",

    # E-cigarettes / vaping
    # Keep: ECIGNOW3 (Has all 3-category current / former / never status)
    "_CURECI3", "MENTCIGS", "MENTECIG",
    
    # Marijuana / cannabis
    # Keep: MARIJAN1 (Give us not just the status but precise amount of consumption days)
    "MARJEAT", "MARJDAB", "MARJOTHR", "MARJSMOK", "MARJVAPE", "USEMRJN4",

    # Alcohol
    # Keep: _DRNKWK3 (Calculated total number of alcoholic beverages consumed per week, 
    # while the alternatives are coarser categories or binary variables)
    "_RFBING6", "DRNKANY6", "ALCDAY4", "DRNK3GE5", 
    "MAXDRNKS", "DROCDY4_", "_RFDRHV9", "AVEDRNK4",

    # Education / Income
    # Keep: EDUCA / INCOME3 (EDUCA and INCOME3 respectively provide a more detailed range of education and income levels)
    "_EDUCAG", "_INCOMG1", 

    # Health insurance / access
    # Keep: PRIMINS2 , PERSDOC3, CHECKUP1
    "MEDCOST1", "_HLTHPL2" , "_HCVU654",

    # Sex
    # Keep: _SEX
    "SEXVAR",

    # Asthma
    # Keep: _ASTHMS1 (Computed Asthma Status, covers all 3 current, former and never asthma status)
    "_LTASTH1", "_CASTHM1", "CASTHDX2", "CASTHNO2", "ASTHNOW", "ASTHMA3", 

    # Cancer / Lung cancer screening test
    # Keep: _LCSCTSN, CERVSCRN, 
    "CSRVDOC1", "CSRVRTRN", "CSRVCLIN", "LCSSCNCR", "LCSCTSC1", "_LCSPSTF",
    "LCSCTWHN", 

    # Dental / oral health
    # Keep: LASTDEN4, RMVTETH4
    "_EXTETH3", "_DENVST3", "_ALTETH3",

    # PSA test follow-ups
    # Keep: PSATEST1 (ever had PSA test)
    "PSATIME1", "PCPSARS2", "PSASUGS1",

    # Mammography / cervical screening
    # Keep: HOWLONG
    "_SGMS102", "_SGMSCP2", "_CLNSCP2", "_MAM402Y", "_RFMAM23",
    "_CRVSCRN", "_RFPAP37",

    # Flu shot
    # FLUSHOT7 (as it has no age restriction, while _FLSHOT7 is only for 65+)
    "_FLSHOT7", "_PNEUMO3", "FLSHTMY3", "IMFVPLA5", "HIVTSTD3",

    # Colorectal screening
    # Keep: COLNSIGM (handles both sigmoidoscopy and colonoscopy, as well as both)
    "_HADSIGM", "_HADCOLN", "VIRCOLO1", "_VIRCOL2", "_RFBLDS6", "_STOLDN2",
    "STOOLDN2", "_CRCREC3", "BLDSTFIT", "SDNATES1", "VCLNTES2", "SMALSTOL",
    "_SBONTI2", "HADSIGM4", "LASTSIG4",
    
    # Arthritis
    # Keep: HAVARTH4
    "_DRDXAR2", "ARTHEXER", "CHKHEMO3", 

    # Urbanicity
    # Keep: _URBSTAT
    "_METSTAT", "MSCODE",

    # HIV/AIDS
    # Keep: HIVTST7
    "_AIDTST4", "_HPV5YR1", "_PAPHPV1", "HPVADSH1",

    # Angina or Coronary Heart Disease
    # Keep: _MICHD and CVDINFR4
    "CVDCRHD4",
], inplace=True)

print("Dataset after dropping redundant features:", df.shape)
df.head(1)

In [ ]:
# Compare the results of correlation analysis after removing redundant features
# from feature_analysis import find_high_correlation_pairs

# pairs_df = find_high_correlation_pairs(df, threshold=0.80, sample_n = 50_000, random_state = 42)
# print(pairs_df.to_string(index=False))

In [ ]:
from feature_groups import build_feature_datasets

datasets = build_feature_datasets(df, target_col)

print('Feature group sizes:', {name: frame.shape[1] - 1 for name, frame in datasets.items()})

for name, ds in datasets.items():
    register_dataset(name, ds, label=name.title())
    print(f"Registered dataset: {name}  shape={ds.shape}")

algorithms = {
    'random_forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'xgboost': XGBClassifier(eval_metric='logloss', random_state=42),
    'lightgbm': LGBMClassifier(random_state=42, verbose=-1),
}

for feat_name, ds in datasets.items():
    X, y = ds.drop(columns=[target_col]), ds[target_col]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    for algo_name, clf in algorithms.items():
        model_id = f"{algo_name}_{feat_name}"
        print(f"Training {model_id}...")

        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        y_prob = clf.predict_proba(X_test)[:, 1]

        pkl_path = (MODELS_DIR / f"{model_id}.pkl").resolve()
        joblib.dump(clf, pkl_path)

        report_raw = classification_report(y_test, y_pred, output_dict=True)
        cm = confusion_matrix(y_test, y_pred)
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        importances = getattr(clf, 'feature_importances_', np.zeros(len(X.columns)))

        register_model(
            model_id=model_id,
            algorithm=algo_name,
            feature_set=feat_name,
            model_path=str(pkl_path),
            metrics={
                'auc': roc_auc_score(y_test, y_prob),
                'accuracy': accuracy_score(y_test, y_pred),
                'f1': f1_score(y_test, y_pred),
                'precision': precision_score(y_test, y_pred),
                'recall': recall_score(y_test, y_pred),
            },
            classification_report={
                'classes': {
                    key: value for key, value in report_raw.items()
                    if key not in ('accuracy', 'macro avg', 'weighted avg')
                },
                'macro_avg': report_raw['macro avg'],
                'weighted_avg': report_raw['weighted avg'],
                'accuracy': report_raw['accuracy'],
            },
            confusion_matrix={
                'tn': int(cm[0, 0]), 'fp': int(cm[0, 1]),
                'fn': int(cm[1, 0]), 'tp': int(cm[1, 1]),
            },
            feature_importances=[
                {'feature': col, 'importance': float(imp)}
                for col, imp in zip(X.columns, importances)
            ],
            roc_curve={'fpr': fpr.tolist(), 'tpr': tpr.tolist()},
            feature_columns=list(X.columns),
        )
        print(f"Registered {model_id}")